# Clawd Chart Foundation 27B

Train a new LoRA adapter on the trainable DavidAU base behind your GGUF release. Two stages: document adaptation, then text/chart-image supervised training. **No model has been trained by generating this notebook.** Select an A100 (40 GB+) runtime before running.

The supplied Colab link is an inference-only demo. This notebook is the separate training workflow. Your current Colab account had A100 disabled during setup; a T4 cannot run this recipe.

## 1. Check the runtime

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select an A100 or larger GPU runtime.'
print(torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory >= 39 * 1024**3, 'At least 40 GB VRAM required for this recipe.'

## 2. Install the training tools
The installed package versions are recorded with the resulting adapter.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==5.16.1', 'peft==0.20.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46,<1', 'pillow==12.3.0'], check=True)

## 3. Upload the prepared data archive
Upload `outputs/chart-foundation-data.zip` from the project. This contains the local datasets, chart images, paper text, and a bounded historical Solarchive sample. Mac paths are not accessible inside Colab. Dataset cards/manifests and unsupported preference records are inventoried rather than mislabeled as supervised examples. Holdouts are kept separate.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, json
archive = Path('/content/chart-foundation-data.zip')
if not archive.exists():
    files.upload()
assert archive.is_file(), 'Upload chart-foundation-data.zip'
data = Path('/content/chart-foundation-data')
data.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        assert (data / item.filename).resolve().is_relative_to(data.resolve()), 'Unsafe archive path'
    z.extractall(data)
manifest = json.loads((data / 'manifest.json').read_text())
print(manifest['counts'])
print(manifest['limitations'])

## 4. Authenticate privately
Use Colab Secrets for `HF_TOKEN`, or an existing Hugging Face login. Do not put tokens in notebook cells.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print('No Colab HF_TOKEN available; public model access will be attempted.')

## 5. Validate and train
The language adapter uses full-sequence loss, with padding and vision markers masked. Vision encoder weights are frozen. Document training precedes supervised training; held-out validation/test records never enter either training split. Default is one complete epoch per stage. Setting a positive step cap creates a limited pilot, not a full run.

In [ ]:
from pathlib import Path
Path('/content/train_chart_foundation.py').write_text('#!/usr/bin/env python3\n"""Two-stage QLoRA adaptation of the DavidAU trainable vision checkpoint."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nBASE = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NM-DAU"\nREVISION = "f3831969c184aa06fdea1bf1060f27a9c9b0eccd"\n\n\ndef read_rows(path):\n    with path.open() as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--data", type=Path, required=True)\n    p.add_argument("--output", type=Path, default=Path("/content/clawd-chart-foundation-27b-lora"))\n    p.add_argument("--epochs", type=float, default=1.0)\n    p.add_argument("--max-steps", type=int, default=-1, help="Per-stage step cap; -1 uses complete epochs")\n    p.add_argument("--preflight", action="store_true")\n    p.add_argument("--smoke", action="store_true", help="One optimizer step per stage on a small mixed image/text subset")\n    p.add_argument("--hub-repo", help="Persist checkpoints to an existing private model repository")\n    args = p.parse_args()\n    manifest = json.loads((args.data / "manifest.json").read_text())\n    sets = {name: read_rows(args.data / f"{name}.jsonl") for name in\n            ("train", "validation", "test", "documents-train", "documents-validation")}\n    if any(not rows for rows in sets.values()):\n        raise ValueError("Every required training/evaluation split must be nonempty")\n    ids = {s: {r["id"] for r in sets[s]} for s in ("train", "validation", "test")}\n    if ids["train"] & (ids["validation"] | ids["test"]):\n        raise ValueError("Training/evaluation duplicate leakage")\n    for split in ("train", "validation", "test"):\n        for row in sets[split]:\n            for image in row["images"]:\n                path = (args.data / image).resolve()\n                if not path.is_relative_to(args.data.resolve()) or not path.is_file():\n                    raise ValueError("Missing or invalid packaged image")\n    print(json.dumps({"base": BASE, "revision": REVISION, "counts": manifest["counts"]}, indent=2))\n    if args.preflight:\n        return\n    if args.smoke:\n        for name, rows in sets.items():\n            images = [r for r in rows if r.get("images")]\n            text = [r for r in rows if not r.get("images")]\n            sets[name] = images[:2] + text[:2]\n        args.max_steps = 1\n\n    import torch\n    from PIL import Image\n    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration, Trainer, TrainerCallback, TrainingArguments, set_seed\n\n    if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 39 * 1024**3:\n        raise RuntimeError("This 27B QLoRA recipe requires a CUDA GPU with at least 40 GB VRAM. Select A100 or larger.")\n    set_seed(42)\n    processor = AutoProcessor.from_pretrained(BASE, revision=REVISION)\n    processor.tokenizer.padding_side = "right"\n    if processor.tokenizer.pad_token_id is None:\n        processor.tokenizer.pad_token = processor.tokenizer.eos_token\n    filtered_counts = {}\n    for name, rows in sets.items():\n        kept = []\n        for row in rows:\n            text = row.get("text")\n            if text is None:\n                text = processor.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)\n            if len(processor.tokenizer.encode(text, add_special_tokens=False)) <= 6144:\n                kept.append(row)\n        filtered_counts[name] = len(rows) - len(kept)\n        sets[name] = kept\n        if not kept:\n            raise ValueError(f"No records fit the token budget in {name}")\n    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(BASE, revision=REVISION,\n        quantization_config=quant, device_map={"": 0}, torch_dtype=torch.bfloat16, attn_implementation="sdpa")\n    model = prepare_model_for_kbit_training(model)\n    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,\n        task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n        exclude_modules=r".*visual.*"))\n    model.config.use_cache = False\n\n    def collate(rows):\n        texts, images = [], []\n        is_document = "text" in rows[0]\n        for row in rows:\n            if is_document:\n                texts.append(row["text"])\n                continue\n            messages = [{**m} for m in row["messages"]]\n            if row["images"]:\n                for msg in messages:\n                    if msg["role"] == "user":\n                        msg["content"] = [{"type": "image"} for _ in row["images"]] + [{"type": "text", "text": msg["content"]}]\n                        break\n                for image in row["images"]:\n                    with Image.open(args.data / image) as im:\n                        im = im.convert("RGB")\n                        im.thumbnail((512, 512))\n                        images.append(im.copy())\n            texts.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))\n        batch = processor(text=texts, images=images or None, padding=True, return_tensors="pt")\n        # Never truncate image-token spans. Oversized records must be filtered before the run.\n        if batch["input_ids"].shape[1] > 8192:\n            raise ValueError("Record exceeds 8192 tokens; filter/rechunk it rather than truncating image tokens")\n        labels = batch["input_ids"].clone()\n        labels[batch["attention_mask"] == 0] = -100\n        for token in ("<|image_pad|>", "<|vision_start|>", "<|vision_end|>"):\n            token_id = processor.tokenizer.get_vocab().get(token)\n            if token_id is not None:\n                labels[labels == token_id] = -100\n        batch["labels"] = labels\n        return batch\n\n    args.output.mkdir(parents=True, exist_ok=True)\n    processor.save_pretrained(args.output / "adapter")\n    (args.output / "data-manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n\n    class PersistCheckpoint(TrainerCallback):\n        def on_save(self, training_args, state, control, **kwargs):\n            if args.hub_repo:\n                from huggingface_hub import HfApi\n                HfApi().upload_folder(repo_id=args.hub_repo, folder_path=str(args.output),\n                    commit_message=f"Persist {Path(training_args.output_dir).name} step {state.global_step}")\n\n    metrics = {"smoke_only": args.smoke, "records_removed_for_length": filtered_counts, "max_steps_per_stage": args.max_steps}\n    for stage, train_key, eval_key in (("documents", "documents-train", "documents-validation"), ("sft", "train", "validation")):\n        training = TrainingArguments(output_dir=str(args.output / stage), per_device_train_batch_size=1,\n            per_device_eval_batch_size=1, gradient_accumulation_steps=16, learning_rate=5e-5,\n            num_train_epochs=args.epochs, max_steps=args.max_steps, bf16=True, gradient_checkpointing=True,\n            gradient_checkpointing_kwargs={"use_reentrant": False}, optim="adamw_torch", logging_steps=10,\n            save_steps=100, save_total_limit=2, eval_strategy="steps", eval_steps=500,\n            remove_unused_columns=False, report_to="none", seed=42)\n        trainer = Trainer(model=model, args=training, train_dataset=sets[train_key],\n                          eval_dataset=sets[eval_key], data_collator=collate, callbacks=[PersistCheckpoint()])\n        metrics[stage] = trainer.train().metrics\n        metrics[stage + "_eval"] = trainer.evaluate()\n        trainer.save_model(str(args.output / "adapter"))\n    metrics["test"] = trainer.evaluate(eval_dataset=sets["test"], metric_key_prefix="test")\n    processor.save_pretrained(args.output / "adapter")\n    (args.output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\\n")\n    (args.output / "data-manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    (args.output / "adapter/README.md").write_text(\n        f"---\\nbase_model: {BASE}\\n---\\n# Clawd Chart Foundation 27B LoRA\\n\\n"\n        f"Smoke-only run: {args.smoke}. Max steps per stage: {args.max_steps}.\\n\\n"\n        "Two-stage document and chart/text adaptation. This is an adapter; load it with the pinned base checkpoint.\\n\\n"\n        f"Base revision: `{REVISION}`. Training uses full-sequence loss with padding and image markers masked. "\n        "Loss metrics do not establish trading profitability. See metrics.json and data-manifest.json for evidence.\\n"\n        "Historical token sample: Data from SolArchive.org (CC BY 4.0).\\n")\n    print("Saved trained adapter and evaluation metrics:", args.output)\n\n\nif __name__ == "__main__":\n    main()\n')
subprocess.run([sys.executable, '/content/train_chart_foundation.py', '--data', str(data), '--preflight'], check=True)

In [ ]:
MAX_STEPS = -1  # -1 = full epoch per stage; a positive value is a pilot run
output = Path('/content/clawd-chart-foundation-27b-lora')
subprocess.run([sys.executable, '/content/train_chart_foundation.py', '--data', str(data), '--output', str(output), '--max-steps', str(MAX_STEPS)], check=True)
(output / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## 6. Review and download the result
The saved artifact is a new adapter, not a merged GGUF. It must be loaded with the pinned trainable base. Evaluation loss measures language modeling fit; it does not establish trading performance. No automatic publication or live trading occurs.

In [ ]:
import shutil
print(json.loads((output / 'metrics.json').read_text()))
assert (output / 'adapter/adapter_model.safetensors').is_file()
archive_out = shutil.make_archive('/content/clawd-chart-foundation-27b-lora', 'zip', output)
files.download(archive_out)